In [1]:
import os
import itertools
from textwrap import dedent
from datetime import datetime
import socket

In [2]:
def mkdir(dir):
    if not os.path.exists(dir):
        os.mkdir(dir)

def detect_cluster() -> str:
    # Prefer explicit SLURM variable, fall back to hostname pattern
    slurm_cluster = os.environ.get("SLURM_CLUSTER_NAME")
    if slurm_cluster:
        return slurm_cluster.lower()
    host = socket.gethostname().lower()
    if "misha" in host:
        return "misha"
    if "bouchet" in host:
        return "bouchet"
    return "misha"  # safe default

cluster = detect_cluster()

print(f"Detected cluster: {cluster}")

Detected cluster: bouchet


In [3]:
# global job parameters

job_directory = f"jobs/fineweb_edu"
out_dir = f'{job_directory}/.out'
job_duration = '00-48:00:00'
partition = "gpu_h200" if cluster == "bouchet" else "gpu"
ntasks = 1
nodes = 1
cpu_per_gpu = 6
mem_per_cpu = 6
# n_gpus = 1
# gpus_constraints = '"h100|a100"' # all gpus are pretty good now

mkdir(job_directory)
mkdir(out_dir)

# PROJECT_ROOT = "/gpfs/radev/home/ma2393/project/Abstractor-HRM"
if cluster == "bouchet":
    project_dir = "/home/ma2393/project_pi_jl2994/ma2393/abstract_transformer/experiments/fineweb"
elif cluster == "misha":
    project_dir = "/home/ma2393/project/abstract_transformer/experiments/fineweb"
else:
    raise ValueError(f"Unknown cluster: {cluster}")

print(f"Setting project_dir to {project_dir} for {cluster} cluster")

Setting project_dir to /home/ma2393/project_pi_jl2994/ma2393/abstract_transformer/experiments/fineweb for bouchet cluster


In [ ]:
# model params
T = 1024
total_batch_size = 524_288

# NOTE: did not add relations?
model_params = [
    # New Hadamard Relational Attention Experiments
    # 350M scale
    dict(d_model=1024, n_layers=24, sa=8, ra=8,
        symbol_type='null',
        # sym_attn_n_symbols=1024, sym_attn_n_heads=8,
        B=8, ra_type='hadamard_relational_attention',
        n_gpus=2, #gpus_constraints= '"h100|a100"',
        param_ct_string='366M'),

    # 750M scale
    dict(d_model=1536, n_layers=24, sa=12, ra=12,
        symbol_type='null',
        # sym_attn_n_symbols=1024, sym_attn_n_heads=8,
        ra_type='hadamard_relational_attention',
        n_gpus=4, B=8, #gpus_constraints= '"h100|a100"',
        param_ct_string='785M'),

    # 1.3B scale
    dict(d_model=2048, n_layers=24, sa=16, ra=16,
        symbol_type='null',
        # sym_attn_n_symbols=1024, sym_attn_n_heads=8,
        ra_type='hadamard_relational_attention', # n_relations=64, 
        n_gpus=4, B=4, #gpus_constraints= '"h100"',
        param_ct_string='1.36B'),

    ## 350M scale
    # dict(d_model=1024, n_layers=24, sa=8, ra=8,
    #     sym_attn_n_symbols=1024, sym_attn_n_heads=8,
    #     n_kv_heads=1, B=8,
    #     n_gpus=2, gpus_constraints= '"h100|a100"',
    #     param_ct_string='325M'),
    # dict(d_model=1024, n_layers=24, sa=8, ra=8,
    #     sym_attn_n_symbols=1024, sym_attn_n_heads=8,
    #     n_kv_heads=2, B=8,
    #     n_gpus=2, gpus_constraints= '"h100|a100"',
    #     param_ct_string='330M'),
    # dict(d_model=1024, n_layers=24, sa=8, ra=8,
    #     sym_attn_n_symbols=1024, sym_attn_n_heads=8,
    #     n_kv_heads=4, B=8,
    #     n_gpus=2, gpus_constraints= '"h100|a100"',
    #     param_ct_string='343M'),
    # dict(d_model=1024, n_layers=24, sa=8, ra=8,
    #     sym_attn_n_symbols=1024, sym_attn_n_heads=8,
    #     n_kv_heads=4, B=8, n_relations=64,
    #     n_gpus=2, gpus_constraints= '"h100|a100"',
    #     param_ct_string='343M'),
    # dict(d_model=1024, n_layers=24, sa=8, ra=8,
    #     sym_attn_n_symbols=1024, sym_attn_n_heads=8,
    #     n_kv_heads=4, B=8, n_relations=32,
    #     n_gpus=2, gpus_constraints= '"h100|a100"',
    #     param_ct_string='343M'),
    # dict(d_model=1024, n_layers=24, sa=8, ra=8,
    #     sym_attn_n_symbols=1024, sym_attn_n_heads=8,
    #     n_kv_heads=4, B=8, symmetric_rels=1, n_relations=32,
    #     n_gpus=2, gpus_constraints= '"h100|a100"',
    #     param_ct_string='332M'),
    # dict(d_model=1024, n_layers=24, sa=8, ra=8,
    #     n_relations=32, sym_attn_n_symbols=1024, sym_attn_n_heads=8,
    #     share_attn_params=1, B=8,
    #     n_gpus=2, gpus_constraints= '"h100|a100"',
    #     param_ct_string='343M'),

    ## 750M scale
    # Transformer
    # dict(d_model=1536, n_layers=24, sa=24, ra=0,
    #     # sym_attn_n_symbols=1024, sym_attn_n_heads=8,
    #     # n_kv_heads=4, symmetric_rels=1, n_relations=32,
    #     n_gpus=2, B=8, gpus_constraints= '"h100|a100"',
    #     param_ct_string='757M'),
    # # DAT
    # dict(d_model=1536, n_layers=24, sa=12, ra=12,
    #     sym_attn_n_symbols=1024, sym_attn_n_heads=8,
    #     n_kv_heads=6, n_relations=64,
    #     n_gpus=4, B=8, gpus_constraints= '"h100|a100"',
    #     param_ct_string='734M'),
    # dict(d_model=1536, n_layers=24, sa=12, ra=12,
    #     sym_attn_n_symbols=1536, sym_attn_n_heads=12,
    #     n_relations=64, # MHA not GQA
    #     n_gpus=4, B=8, gpus_constraints= '"h100|a100"', # need to lower B=8 to say B=6 or 4
    #     param_ct_string='794M'), # Not submitted yet

    ## 1.3B scale
    # dict(d_model=2048, n_layers=24, sa=16, ra=16,
    #     n_relations=64, sym_attn_n_symbols=1024, sym_attn_n_heads=8,
    #     share_attn_params=1, B=4,
    #     n_gpus=4, gpus_constraints= '"h100"',
    #     param_ct_string='1.27B'),

    # dict(d_model=2048, n_layers=24, sa=16, ra=16,
    #     n_relations=64, sym_attn_n_symbols=1024, sym_attn_n_heads=8,
    #     B=4, n_kv_heads=4,
    #     n_gpus=4, gpus_constraints= '"h100"',
    #     param_ct_string='1.22B'),
    # dict(d_model=2048, n_layers=24, sa=16, ra=16,
    #     n_relations=64,  # sym_attn_n_symbols=1024, sym_attn_n_heads=8,
    #     B=4, n_kv_heads=8,
    #     n_gpus=4, gpus_constraints= '"h100"',
    #     param_ct_string='1.27B'),
    # dict(d_model=2048, n_layers=24, sa=16, ra=16,
    #     n_relations=128, sym_attn_n_symbols=512, sym_attn_n_heads=16,
    #     B=2, n_kv_heads=8,
    #     n_gpus=4, gpus_constraints= '"h100"',
    #     param_ct_string='1.27B'),
    # dict(d_model=2048, n_layers=24, sa=16, ra=16,
    #     n_relations=128, sym_attn_n_symbols=2048, sym_attn_n_heads=16,
    #     B=2, n_kv_heads=8,
    #     n_gpus=4, gpus_constraints= '"h100"',
    #     param_ct_string='1.27B'),
    # dict(d_model=2048, n_layers=24, sa=16, ra=16,
    #     n_relations=64,  sym_attn_n_symbols=512, sym_attn_n_heads=16,
    #     B=4, n_kv_heads=8,
    #     n_gpus=4, gpus_constraints= '"h100"',
    #     param_ct_string='1.27B'),
    # # experiment with symbolic attention params here w/ full 1.37B model (later, should experiment w/ trimmed models)
    # dict(d_model=2048, n_layers=24, sa=16, ra=16,
    #     n_relations=64, sym_attn_n_symbols=2048, sym_attn_n_heads=8,
    #     B=4,
    #     n_gpus=4, gpus_constraints= '"h100"',
    #     param_ct_string='1.37B'),
    # dict(d_model=2048, n_layers=24, sa=16, ra=16,
    #     n_relations=64,
    #     shared_symbol_retriever=0, weight_tie_symbol_library=1, trainable_symbols=1,
    #     B=4, n_kv_heads=4,
    #     n_gpus=4, gpus_constraints= '"h100"',
    #     param_ct_string='1.42B'),
]

jobs_params = []
for mparams in model_params:
    # compute run name
    if mparams['ra'] > 0:
        run_name = f"DAT-sa{mparams['sa']}-ra{mparams['ra']}"
        if 'ra_type' in mparams:
            run_name += f"-{mparams['ra_type']}"
        if 'n_relations' in mparams:
            run_name += f"-nr{mparams['n_relations']}"
        if 'symmetric_rels' in mparams:
            run_name += f"-sr{mparams['symmetric_rels']}"
        if 'share_attn_params' in mparams:
            run_name += f"-sharedattn{mparams['share_attn_params']}"
        if 'sym_attn_n_symbols' in mparams:
            run_name += f"-ns{mparams['sym_attn_n_symbols']}"
        if 'sym_attn_n_heads' in mparams:
            run_name += f"-sh{mparams['sym_attn_n_heads']}"
        if 'shared_symbol_retriever' in mparams:
            run_name += f"-ssr{mparams['shared_symbol_retriever']}"
        if 'weight_tie_symbol_library' in mparams:
            run_name += f"-wt{mparams['weight_tie_symbol_library']}"
        if 'trainable_symbols' in mparams:
            run_name += f"-ts{mparams['trainable_symbols']}"
    else:
        run_name = f'T-sa{mparams["sa"]}'
    if 'n_kv_heads' in mparams:
        run_name += f'-nkvh{mparams["n_kv_heads"]}'
    if 'param_ct_string' in mparams:
        run_name += f'-{mparams["param_ct_string"]}'

    jobs_params.append({**mparams, 'run_name': run_name})

In [5]:
jobs_params

[{'d_model': 1024,
  'n_layers': 24,
  'sa': 8,
  'ra': 8,
  'symbol_type': 'null',
  'B': 8,
  'ra_type': 'hadamard_relational_attention',
  'n_gpus': 2,
  'gpus_constraints': '"h100|a100"',
  'param_ct_string': '366M',
  'run_name': 'DAT-sa8-ra8-hadamard_relational_attention-366M'},
 {'d_model': 1536,
  'n_layers': 24,
  'sa': 12,
  'ra': 12,
  'symbol_type': 'null',
  'ra_type': 'hadamard_relational_attention',
  'n_gpus': 4,
  'B': 8,
  'gpus_constraints': '"h100|a100"',
  'param_ct_string': '734M',
  'run_name': 'DAT-sa12-ra12-hadamard_relational_attention-734M'},
 {'d_model': 2048,
  'n_layers': 24,
  'sa': 16,
  'ra': 16,
  'symbol_type': 'null',
  'ra_type': 'hadamard_relational_attention',
  'n_gpus': 4,
  'B': 4,
  'gpus_constraints': '"h100"',
  'param_ct_string': '1.27B',
  'run_name': 'DAT-sa16-ra16-hadamard_relational_attention-1.27B'}]

In [6]:
len(jobs_params)

3

In [7]:
# global config parameters
n_epochs = 1
max_steps = -1
log_to_wandb = 1

In [8]:
QOS = "qos_nmi" if cluster == "misha" else None
date_str = datetime.now().strftime("%Y-%m-%d")
gpu_type = "h200" if cluster == "bouchet" else "h100"

In [9]:
# create jobs
created_jobs = []
for params in jobs_params:

    job_file = os.path.join(job_directory, f"{params['run_name']}.job")

    with open(job_file, 'w') as fh:
        fh.writelines(f"#!/bin/bash\n")
        fh.writelines(f"#SBATCH --job-name={params['run_name']}.{date_str}\n")
        fh.writelines(f"#SBATCH --partition={partition}\n")
        fh.writelines(f"#SBATCH --output={out_dir}/%j-{params['run_name']}.out\n")
        fh.writelines(f"#SBATCH --ntasks={ntasks} --nodes={nodes}\n")
        fh.writelines(f"#SBATCH --cpus-per-gpu={cpu_per_gpu}\n")
        fh.writelines(f"#SBATCH --mem-per-cpu={mem_per_cpu}G\n")
        fh.writelines(f"#SBATCH --time={job_duration}\n")
        fh.writelines(f"#SBATCH --mail-type=ALL\n")
        fh.writelines(f"#SBATCH --gpus={gpu_type}:{params['n_gpus']}\n")
        if QOS:
            fh.writelines(f"#SBATCH --qos={QOS}\n")

        fh.writelines('\n')
        fh.writelines('module load StdEnv\n')
        fh.writelines('export SLURM_EXPORT_ENV=ALL\n')
        fh.writelines('\n')

        # fh.writelines(f"module restore python_env\n") # load modules i need
        fh.writelines(f"module load miniconda\n") # load modules i need
        # fh.writelines(f"conda init\n")
        fh.writelines(f"conda activate abstractor_hrm\n") # activate conda environment
        fh.writelines(f"conda info --envs\n") # activate conda environment

        fh.writelines('\n')
        fh.writelines(f"nvidia-smi -L\n") # print gpu information
        fh.writelines('\n')

        fh.writelines(f"cd {project_dir}\n") # navigate to project directory
        fh.writelines('\n')

        # run python script
        if params['n_gpus'] > 1:
            fh.writelines(f"torchrun --standalone --nproc_per_node={params['n_gpus']} pretrain.py \\\n")
        else:
            fh.writelines(f"python pretrain.py \\\n")

        fh.writelines(f"\t--d_model {params['d_model']} --sa {params['sa']} --ra {params['ra']} --n_layers {params['n_layers']} \\\n")
        if 'ra_type' in params:
            fh.writelines(f"\t--ra_type {params['ra_type']} \\\n")
        if 'symbol_type' in params:
            fh.writelines(f"\t--symbol_type {params['symbol_type']} \\\n")
        if 'n_relations' in params:
            fh.writelines(f"\t--n_relations {params['n_relations']} \\\n")
        if 'share_attn_params' in params:
            fh.writelines(f"\t--share_attn_params {params['share_attn_params']} \\\n")
        if 'symmetric_rels' in params:
            fh.writelines(f"\t--symmetric_rels {params['symmetric_rels']} \\\n")
        if 'sym_attn_n_symbols' in params:
            fh.writelines(f"\t--sym_attn_n_symbols {params['sym_attn_n_symbols']} --sym_attn_n_heads {params['sym_attn_n_heads']} \\\n")
        if 'n_kv_heads' in params:
            fh.writelines(f"\t--n_kv_heads {params['n_kv_heads']} \\\n")
        if 'shared_symbol_retriever' in params:
            fh.writelines(f"\t--shared_symbol_retriever {params['shared_symbol_retriever']} --weight_tie_symbol_library {params['weight_tie_symbol_library']} ")
            fh.writelines(f"--trainable_symbols {params['trainable_symbols']} \\\n")
        fh.writelines(f"\t--T {T} --B {params['B']} --total_batch_size {total_batch_size} \\\n")
        fh.writelines(f"\t--wandb_log 1 --run_name {params['run_name']} --job_duration {job_duration} \\\n")

    created_jobs.append(job_file)

In [10]:
created_jobs

['jobs/fineweb_edu/DAT-sa8-ra8-hadamard_relational_attention-366M.job',
 'jobs/fineweb_edu/DAT-sa12-ra12-hadamard_relational_attention-734M.job',
 'jobs/fineweb_edu/DAT-sa16-ra16-hadamard_relational_attention-1.27B.job']

In [11]:
confirm = input("CONTINUE TO RUN ALL JOBS?")
if confirm == 'y':
    for job in created_jobs:
        os.system(f'sbatch {job}')
else:
    print('JOBS NOT SUBMITTED')

JOBS NOT SUBMITTED
